# NLP Project 2 – Fehleranalyse & Visualisierung
Whisper v1 / v2 / v3 auf STT4SG-350 (Schweizerdeutsch)

**Ablauf:**
1. CSVs laden & Spalten prüfen
2. WER & CER berechnen (overall + pro Dialektregion)
3. Fehlertypen analysieren (Substitutionen, Deletions, Insertions)
4. Qualitative Fehlerbeispiele
5. Visualisierungen für Poster generieren

## 0. Setup

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import os
from collections import Counter

# jiwer für WER/CER
# falls nicht installiert: pip install jiwer
from jiwer import process_words, process_characters

matplotlib.rcParams['figure.dpi'] = 150
os.makedirs('figures', exist_ok=True)

print('Setup complete.')

Setup complete.


## 1. CSVs laden & Spalten prüfen

⚠️ **Hier zuerst die Spalten anschauen und bei Bedarf `COL_REFERENCE` / `COL_HYPOTHESIS` / `COL_REGION` anpassen!**

In [3]:
RESULTS_DIR = 'results/'

# Pfade zu den Checkpoint-CSVs
CSV_FILES = {
    'v1': os.path.join(RESULTS_DIR, 'checkpoint_openai_whisper-large.csv'),
    'v2': os.path.join(RESULTS_DIR, 'checkpoint_openai_whisper-large-v2.csv'),
    'v3': os.path.join(RESULTS_DIR, 'checkpoint_openai_whisper-large-v3.csv'),
}

# CSVs laden
dfs = {}
for version, path in CSV_FILES.items():
    df = pd.read_csv(path)
    dfs[version] = df
    print(f'\n=== Whisper {version} ===')
    print(f'Shape: {df.shape}')
    print(f'Spalten: {df.columns.tolist()}')
    print(df.head(2))


=== Whisper v1 ===
Shape: (24605, 5)
Spalten: ['path', 'sentence', 'canton', 'model', 'hypothesis']
                                                path  \
0  8800b4de-fa22-4fdc-9f29-457c4010fd57/d57790928...   
1  887b50f8-215b-4a1d-8f32-13516da6506f/73ca6c9b0...   

                                            sentence canton  \
0  Die Steuerfrage wäre eine andere, die Planung ...     BS   
1  Ich bin der Meinung, dass ihr noch nicht alle ...     BE   

                  model                                         hypothesis  
0  openai/whisper-large  Die Steuerfragen wären andere, die Planungen a...  
1  openai/whisper-large  Ich bin der Meinung, dass Sie noch nicht alle ...  

=== Whisper v2 ===
Shape: (24605, 5)
Spalten: ['path', 'sentence', 'canton', 'model', 'hypothesis']
                                                path  \
0  8800b4de-fa22-4fdc-9f29-457c4010fd57/d57790928...   
1  887b50f8-215b-4a1d-8f32-13516da6506f/73ca6c9b0...   

                                       

In [11]:
df[["sentence", "hypothesis"]]

,sentence,hypothesis
0,"Die Steuerfrage wäre eine andere, die Planung ...","Die Steuerfrage wären andere, die Planungen na..."
1,"Ich bin der Meinung, dass ihr noch nicht alle ...","Ich bin der Meinung, dass ihr noch nicht alle ..."
2,Dann änderte sie ihre Meinung und verlangte di...,Dann hat sie ihre Meinung geändert und den Tau...
3,Und neue Türen haben sich nicht geöffnet.,Und neue Türen haben sich nicht geöffnet.
4,Der Gemeinderat nimmt es auch als Postulat ent...,Der Gemeindetrode nimmt Sauer das Postulat ent...
...,...,...
24600,Viele sind völlig darauf fixiert.,Viele sind völlig darauf fixiert.
24601,Was ist denn das für eine Politik?,Was ist denn das für eine Politik?
24602,An den Häfen drohten Chaos und Staus.,An der Hefe drohen Chaos und Staus.
24603,Diese Anträge fanden keine Mehrheit.,Die Anträge haben keine Mehrheit gefunden.


In [13]:
# ⚠️ ANPASSEN falls Spaltenname anders ist!
COL_REFERENCE  = 'sentence'        # Ground Truth (aus test.tsv)
COL_HYPOTHESIS = 'hypothesis' # Whisper Output
COL_REGION     = 'canton'       # Dialektregion

# Kurzer Check
for version, df in dfs.items():
    for col in [COL_REFERENCE, COL_HYPOTHESIS, COL_REGION]:
        assert col in df.columns, f'Spalte "{col}" nicht in {version}! Verfügbar: {df.columns.tolist()}'
print('Alle Spaltennamen OK ✓')

Alle Spaltennamen OK ✓


## 2. WER & CER berechnen

In [14]:
def safe_str(text):
    """NaN-sichere Konvertierung zu String."""
    if pd.isna(text):
        return ''
    return str(text).strip().lower()


def compute_wer_cer(df, col_ref=COL_REFERENCE, col_hyp=COL_HYPOTHESIS):
    """Berechnet WER und CER für einen DataFrame."""
    refs = [safe_str(t) for t in df[col_ref]]
    hyps = [safe_str(t) for t in df[col_hyp]]
    
    wer_result = process_words(refs, hyps)
    cer_result = process_characters(refs, hyps)
    
    total_words = wer_result.hits + wer_result.substitutions + wer_result.deletions
    
    return {
        'wer':           wer_result.wer,
        'cer':           cer_result.cer,
        'substitutions': wer_result.substitutions,
        'deletions':     wer_result.deletions,
        'insertions':    wer_result.insertions,
        'hits':          wer_result.hits,
        'total_words':   total_words,
        'n_samples':     len(df),
    }


# Overall WER/CER
print('=== Overall WER / CER ===' )
overall_results = {}
for version, df in dfs.items():
    result = compute_wer_cer(df)
    overall_results[version] = result
    print(f'Whisper {version}: WER={result["wer"]:.3f} | CER={result["cer"]:.3f} | '
          f'Sub={result["substitutions"]} Del={result["deletions"]} Ins={result["insertions"]}')

=== Overall WER / CER ===
Whisper v1: WER=0.294 | CER=0.143 | Sub=42764 Del=6739 Ins=7979
Whisper v2: WER=0.256 | CER=0.125 | Sub=37384 Del=5839 Ins=6964
Whisper v3: WER=0.250 | CER=0.121 | Sub=36599 Del=5832 Ins=6466


In [15]:
# WER / CER pro Dialektregion
REGIONS = sorted(dfs['v1'][COL_REGION].dropna().unique())
print(f'Dialektregionen: {REGIONS}\n')

region_results = {}  # region_results[version][region] = {wer, cer, ...}

for version, df in dfs.items():
    region_results[version] = {}
    for region in REGIONS:
        df_r = df[df[COL_REGION] == region]
        result = compute_wer_cer(df_r)
        region_results[version][region] = result

# Als DataFrame ausgeben
rows = []
for version in ['v1', 'v2', 'v3']:
    for region in REGIONS:
        r = region_results[version][region]
        rows.append({
            'Modell': f'whisper-large-{version}',
            'Region': region,
            'WER':    round(r['wer'], 4),
            'CER':    round(r['cer'], 4),
            'N':      r['n_samples'],
        })

df_region = pd.DataFrame(rows)
print(df_region.to_string(index=False))

# Speichern
df_region.to_csv('results/wer_by_region.csv', index=False)
print('\n→ Gespeichert: results/wer_by_region.csv')

Dialektregionen: ['AG', 'BE', 'BL', 'BS', 'GL', 'GR', 'LU', 'SG', 'SH', 'SO', 'TG', 'TI', 'UR', 'VS', 'ZG', 'ZH']

          Modell Region    WER    CER    N
whisper-large-v1     AG 0.2589 0.1245 1550
whisper-large-v1     BE 0.3464 0.1814 2494
whisper-large-v1     BL 0.3480 0.1637 1254
whisper-large-v1     BS 0.2982 0.1368 2261
whisper-large-v1     GL 0.2811 0.1373  364
whisper-large-v1     GR 0.2658 0.1322 3159
whisper-large-v1     LU 0.2503 0.1207 1704
whisper-large-v1     SG 0.2869 0.1362 2450
whisper-large-v1     SH 0.1870 0.0908  344
whisper-large-v1     SO 0.3341 0.1641  813
whisper-large-v1     TG 0.2767 0.1529  721
whisper-large-v1     TI 0.2030 0.0985  356
whisper-large-v1     UR 0.2184 0.1001  365
whisper-large-v1     VS 0.3794 0.1843 3515
whisper-large-v1     ZG 0.2064 0.0994  723
whisper-large-v1     ZH 0.2396 0.1147 2532
whisper-large-v2     AG 0.2228 0.1085 1550
whisper-large-v2     BE 0.3020 0.1593 2494
whisper-large-v2     BL 0.3168 0.1500 1254
whisper-large-v2     BS 0

In [16]:
# Overall-Tabelle speichern
overall_rows = []
for version in ['v1', 'v2', 'v3']:
    r = overall_results[version]
    overall_rows.append({
        'Modell': f'whisper-large-{version}',
        'WER':    round(r['wer'], 4),
        'CER':    round(r['cer'], 4),
        'N':      r['n_samples'],
    })
df_overall = pd.DataFrame(overall_rows)
df_overall.to_csv('results/wer_overall.csv', index=False)
print('→ Gespeichert: results/wer_overall.csv')
print(df_overall.to_string(index=False))

→ Gespeichert: results/wer_overall.csv
          Modell    WER    CER     N
whisper-large-v1 0.2936 0.1432 24605
whisper-large-v2 0.2564 0.1253 24605
whisper-large-v3 0.2498 0.1213 24605


## 3. Fehlertypen-Analyse (Substitutionen / Deletions / Insertions)

Die WER setzt sich aus 3 Fehlertypen zusammen (auf Wortebene):

| Typ | Beschreibung | Beispiel |
|---|---|---|
| **Substitution** | falsches Wort | `"wäre"` → `"wären"` |
| **Deletion** | Wort fehlt | `"bin"` → *(leer)* |
| **Insertion** | extra Wort eingefügt | *(leer)* → `"neuen"` |

Für CER gilt dasselbe Prinzip auf Zeichenebene.

In [17]:
print('=== Fehlertypen-Verteilung (Overall) ===')
print(f'{"Modell":<20} {"Sub%":>8} {"Del%":>8} {"Ins%":>8}')
print('-' * 46)

error_type_overall = {}
for version in ['v1', 'v2', 'v3']:
    r = overall_results[version]
    total_errors = r['substitutions'] + r['deletions'] + r['insertions']
    if total_errors == 0:
        sub_pct = del_pct = ins_pct = 0
    else:
        sub_pct = r['substitutions'] / total_errors
        del_pct = r['deletions']     / total_errors
        ins_pct = r['insertions']    / total_errors
    
    error_type_overall[version] = {
        'substitutions_pct': sub_pct,
        'deletions_pct':     del_pct,
        'insertions_pct':    ins_pct,
    }
    print(f'whisper-large-{version:<6} {sub_pct*100:>7.1f}% {del_pct*100:>7.1f}% {ins_pct*100:>7.1f}%')

=== Fehlertypen-Verteilung (Overall) ===
Modell                   Sub%     Del%     Ins%
----------------------------------------------
whisper-large-v1        74.4%    11.7%    13.9%
whisper-large-v2        74.5%    11.6%    13.9%
whisper-large-v3        74.8%    11.9%    13.2%


In [18]:
# Fehlertypen pro Dialektregion
print('=== Fehlertypen pro Region (Sub% | Del% | Ins%) ===')

error_type_region = {}
for version in ['v1', 'v2', 'v3']:
    error_type_region[version] = {}
    print(f'\n--- Whisper {version} ---')
    for region in REGIONS:
        r = region_results[version][region]
        total_errors = r['substitutions'] + r['deletions'] + r['insertions']
        if total_errors == 0:
            sub_pct = del_pct = ins_pct = 0
        else:
            sub_pct = r['substitutions'] / total_errors
            del_pct = r['deletions']     / total_errors
            ins_pct = r['insertions']    / total_errors
        error_type_region[version][region] = {
            'substitutions_pct': sub_pct,
            'deletions_pct':     del_pct,
            'insertions_pct':    ins_pct,
        }
        print(f'  {region:<15}: Sub={sub_pct*100:.1f}% Del={del_pct*100:.1f}% Ins={ins_pct*100:.1f}%')

=== Fehlertypen pro Region (Sub% | Del% | Ins%) ===

--- Whisper v1 ---
  AG             : Sub=74.4% Del=13.0% Ins=12.6%
  BE             : Sub=72.9% Del=13.2% Ins=13.9%
  BL             : Sub=77.0% Del=10.8% Ins=12.2%
  BS             : Sub=77.1% Del=10.8% Ins=12.1%
  GL             : Sub=73.7% Del=15.1% Ins=11.2%
  GR             : Sub=73.0% Del=11.7% Ins=15.2%
  LU             : Sub=73.5% Del=12.8% Ins=13.7%
  SG             : Sub=74.5% Del=11.9% Ins=13.6%
  SH             : Sub=76.5% Del=9.2% Ins=14.3%
  SO             : Sub=75.4% Del=12.4% Ins=12.2%
  TG             : Sub=70.8% Del=11.3% Ins=17.9%
  TI             : Sub=75.0% Del=9.3% Ins=15.7%
  UR             : Sub=75.8% Del=11.8% Ins=12.4%
  VS             : Sub=74.2% Del=10.9% Ins=14.9%
  ZG             : Sub=73.4% Del=11.9% Ins=14.7%
  ZH             : Sub=75.1% Del=10.9% Ins=14.0%

--- Whisper v2 ---
  AG             : Sub=72.9% Del=13.7% Ins=13.4%
  BE             : Sub=72.2% Del=13.0% Ins=14.8%
  BL             : Sub=76.4%

## 4. Qualitative Fehlerbeispiele (häufigste falsch erkannte Wörter)

In [ ]:
def get_substitution_pairs(df, col_ref=COL_REFERENCE, col_hyp=COL_HYPOTHESIS, top_n=20):
    """Gibt die häufigsten Substitutions-Paare (ref → hyp) zurück."""
    counter = Counter()
    
    for _, row in df.iterrows():
        ref = safe_str(row[col_ref])
        hyp = safe_str(row[col_hyp])
        if not ref or not hyp:
            continue
        
        result = process_words([ref], [hyp])
        ref_words = ref.split()
        hyp_words = hyp.split()
        
        for chunk in result.alignments[0]:
            if chunk.type == 'substitute':
                ref_word = ' '.join(ref_words[chunk.ref_start_idx:chunk.ref_end_idx])
                hyp_word = ' '.join(hyp_words[chunk.hyp_start_idx:chunk.hyp_end_idx])
                counter[(ref_word, hyp_word)] += 1
    
    return counter.most_common(top_n)


for version in ['v1', 'v2', 'v3']:
    print(f'\n=== Top-20 Substitutionen – Whisper {version} ===')
    pairs = get_substitution_pairs(dfs[version])
    print(f'{"Ref (Ground Truth)":<25} → {"Hyp (Whisper)":<25} {"Anzahl":>6}')
    print('-' * 60)
    for (ref_w, hyp_w), count in pairs:
        print(f'{ref_w:<25} → {hyp_w:<25} {count:>6}')

## 5. Visualisierungen für Poster

In [ ]:
# ── Fig 1: WER Overall (Bar Chart) ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))

versions = ['v1', 'v2', 'v3']
labels   = ['Whisper\nlarge-v1', 'Whisper\nlarge-v2', 'Whisper\nlarge-v3']
wers     = [overall_results[v]['wer'] * 100 for v in versions]
cers     = [overall_results[v]['cer'] * 100 for v in versions]

x = np.arange(len(versions))
width = 0.35
colors_wer = ['#2196F3', '#1565C0', '#0D47A1']
colors_cer = ['#90CAF9', '#64B5F6', '#42A5F5']

bars1 = ax.bar(x - width/2, wers, width, label='WER', color=colors_wer, edgecolor='white')
bars2 = ax.bar(x + width/2, cers, width, label='CER', color=colors_cer, edgecolor='white')

ax.bar_label(bars1, fmt='%.1f%%', padding=3, fontsize=9)
ax.bar_label(bars2, fmt='%.1f%%', padding=3, fontsize=9)

ax.set_ylabel('Fehlerrate (%)')
ax.set_title('Overall WER & CER – Whisper v1/v2/v3')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
ax.set_ylim(0, max(wers + cers) * 1.25)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('figures/fig_wer_overall.png', dpi=200)
plt.show()
print('→ Gespeichert: figures/fig_wer_overall.png')

In [ ]:
# ── Fig 2: WER pro Dialektregion (Grouped Bar Chart) ────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

n_regions = len(REGIONS)
x = np.arange(n_regions)
width = 0.25
version_colors = {'v1': '#1976D2', 'v2': '#388E3C', 'v3': '#D32F2F'}
version_labels = {'v1': 'Whisper large-v1', 'v2': 'Whisper large-v2', 'v3': 'Whisper large-v3'}

for i, version in enumerate(['v1', 'v2', 'v3']):
    wer_per_region = [region_results[version][r]['wer'] * 100 for r in REGIONS]
    offset = (i - 1) * width
    bars = ax.bar(x + offset, wer_per_region, width,
                  label=version_labels[version],
                  color=version_colors[version],
                  edgecolor='white', alpha=0.9)
    ax.bar_label(bars, fmt='%.0f%%', padding=2, fontsize=7)

ax.set_ylabel('WER (%)')
ax.set_title('WER pro Dialektregion – Whisper v1 / v2 / v3')
ax.set_xticks(x)
ax.set_xticklabels(REGIONS, rotation=20, ha='right')
ax.legend(loc='upper right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('figures/fig_wer_by_region.png', dpi=200)
plt.show()
print('→ Gespeichert: figures/fig_wer_by_region.png')

In [ ]:
# ── Fig 3: Fehlertypen Overall (Stacked Bar) ────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))

versions_labels = ['Whisper\nlarge-v1', 'Whisper\nlarge-v2', 'Whisper\nlarge-v3']
subs = [error_type_overall[v]['substitutions_pct'] * 100 for v in ['v1', 'v2', 'v3']]
dels = [error_type_overall[v]['deletions_pct']     * 100 for v in ['v1', 'v2', 'v3']]
ins  = [error_type_overall[v]['insertions_pct']    * 100 for v in ['v1', 'v2', 'v3']]

x = np.arange(3)
p1 = ax.bar(x, subs, label='Substitutionen', color='#E53935', edgecolor='white')
p2 = ax.bar(x, dels, bottom=subs, label='Deletions', color='#FB8C00', edgecolor='white')
p3 = ax.bar(x, ins,  bottom=[s+d for s,d in zip(subs, dels)], label='Insertions', color='#43A047', edgecolor='white')

ax.set_ylabel('Anteil an Gesamtfehlern (%)')
ax.set_title('Fehlertypen-Verteilung – Whisper v1/v2/v3')
ax.set_xticks(x)
ax.set_xticklabels(versions_labels)
ax.set_ylim(0, 110)
ax.legend(loc='upper right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Labels in Segmente
for i in range(3):
    if subs[i] > 5: ax.text(i, subs[i]/2,            f'{subs[i]:.1f}%', ha='center', va='center', color='white', fontsize=9, fontweight='bold')
    if dels[i] > 5: ax.text(i, subs[i]+dels[i]/2,    f'{dels[i]:.1f}%', ha='center', va='center', color='white', fontsize=9, fontweight='bold')
    if ins[i]  > 5: ax.text(i, subs[i]+dels[i]+ins[i]/2, f'{ins[i]:.1f}%', ha='center', va='center', color='white', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('figures/fig_error_types.png', dpi=200)
plt.show()
print('→ Gespeichert: figures/fig_error_types.png')

In [ ]:
# ── Fig 4: CER pro Dialektregion ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

for i, version in enumerate(['v1', 'v2', 'v3']):
    cer_per_region = [region_results[version][r]['cer'] * 100 for r in REGIONS]
    offset = (i - 1) * width
    bars = ax.bar(x + offset, cer_per_region, width,
                  label=version_labels[version],
                  color=version_colors[version],
                  edgecolor='white', alpha=0.9)
    ax.bar_label(bars, fmt='%.0f%%', padding=2, fontsize=7)

ax.set_ylabel('CER (%)')
ax.set_title('CER pro Dialektregion – Whisper v1 / v2 / v3')
ax.set_xticks(x)
ax.set_xticklabels(REGIONS, rotation=20, ha='right')
ax.legend(loc='upper right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('figures/fig_cer_by_region.png', dpi=200)
plt.show()
print('→ Gespeichert: figures/fig_cer_by_region.png')

## 6. Zusammenfassung

In [ ]:
print('=' * 60)
print('ZUSAMMENFASSUNG')
print('=' * 60)

for version in ['v1', 'v2', 'v3']:
    r = overall_results[version]
    print(f'\nWhisper large-{version}:')
    print(f'  WER:           {r["wer"]*100:.2f}%')
    print(f'  CER:           {r["cer"]*100:.2f}%')
    print(f'  Substitutions: {r["substitutions"]:>6}')
    print(f'  Deletions:     {r["deletions"]:>6}')
    print(f'  Insertions:    {r["insertions"]:>6}')
    
    best_region = min(REGIONS, key=lambda r_: region_results[version][r_]['wer'])
    worst_region = max(REGIONS, key=lambda r_: region_results[version][r_]['wer'])
    print(f'  Beste Region:  {best_region} (WER={region_results[version][best_region]["wer"]*100:.1f}%)')
    print(f'  Schlechteste:  {worst_region} (WER={region_results[version][worst_region]["wer"]*100:.1f}%)')

print('\nGenerierte Figures:')
for f in os.listdir('figures'):
    print(f'  figures/{f}')